In [ ]:
import pandas as pd
import numpy as np

# ============================================
# STEP 1: Caricamento e analisi iniziale
# ============================================
df = pd.read_csv("employees.csv")
print("=== DIMENSIONI DATASET ===")
print(f"Righe: {df.shape[0]}, Colonne: {df.shape[1]}")
print(f"\nColonne: {list(df.columns)}")

# ============================================
# STEP 2: Identificazione dati sbagliati
# ============================================
print("\n=== ANALISI VALORI MANCANTI E ERRATI ===")

# Standardizzare placeholder come NaN
na_placeholders = ['NA', 'na', 'NaN', 'n.a.', '?', '']
for col in df.columns:
    mask = df[col].astype(str).str.strip().isin(na_placeholders)
    count = mask.sum()
    if count > 0:
        print(f"{col}: {count} valori placeholder trovati")
        problematic = df.loc[mask, col].unique()
        print(f"  Valori: {problematic}")

# Errori di battitura in Senior Management
sm_values = df['Senior Management'].dropna().unique()
valid_sm = ['TRUE', 'FALSE']
typos_sm = [v for v in sm_values if v not in valid_sm
            and str(v) not in na_placeholders]
print(f"\nErrori battitura Senior Management: {typos_sm}")

# Errori in Gender
gen_values = df['Gender'].dropna().unique()
valid_gen = ['Male', 'Female']
typos_gen = [v for v in gen_values if v not in valid_gen
             and str(v) not in na_placeholders]
print(f"Errori battitura Gender: {typos_gen}")

# ============================================
# STEP 3: Correzione errori
# ============================================
print("\n=== CORREZIONE ERRORI ===")

# Converti placeholder in NaN
for col in df.columns:
    df[col] = df[col].replace(na_placeholders, np.nan)
    df[col] = df[col].apply(
        lambda x: np.nan if isinstance(x, str)
        and x.strip() in na_placeholders else x
    )

# Correggi errori di battitura
corrections = {
    'Senior Management': {
        'TRIE': 'TRUE', 'TRU': 'TRUE', 'RUE': 'TRUE',
        'FILSE': 'FALSE', 'FOLSE': 'FALSE'
    }
}
for col, mapping in corrections.items():
    df[col] = df[col].replace(mapping)
    print(f"Corretti errori in {col}")

# ============================================
# STEP 4: Conversione tipi di dati
# ============================================
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')
df['Bonus %'] = pd.to_numeric(df['Bonus %'], errors='coerce')
print("Tipi di dati convertiti")

# ============================================
# STEP 5: Imputazione valori mancanti
# ============================================
print("\n=== IMPUTAZIONE VALORI MANCANTI ===")

# Prima imputa Team (serve per le imputazioni raggruppate)
team_mode = df['Team'].mode()[0]
n_team = df['Team'].isnull().sum()
df['Team'] = df['Team'].fillna(team_mode)
print(f"Team: {n_team} valori imputati con '{team_mode}'")

# Salary per gruppo Team
n_sal = df['Salary'].isnull().sum()
df['Salary'] = df.groupby('Team')['Salary'].transform(
    lambda x: x.fillna(x.median())
)
print(f"Salary: {n_sal} valori imputati con mediana per Team")

# Bonus mediana generale
n_bon = df['Bonus %'].isnull().sum()
bonus_median = df['Bonus %'].median()
df['Bonus %'] = df['Bonus %'].fillna(bonus_median)
print(f"Bonus: {n_bon} valori imputati con mediana {bonus_median:.3f}")

# Gender moda
n_gen = df['Gender'].isnull().sum()
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
print(f"Gender: {n_gen} valori imputati")

# Senior Management per gruppo Team
n_sm = df['Senior Management'].isnull().sum()
df['Senior Management'] = df.groupby('Team')[
    'Senior Management'
].transform(
    lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else 'FALSE')
)
print(f"Senior Management: {n_sm} valori imputati")

# First Name
n_fn = df['First Name'].isnull().sum()
df['First Name'] = df['First Name'].fillna('Unknown')
print(f"First Name: {n_fn} valori imputati con 'Unknown'")

# ============================================
# STEP 6: Validazione finale
# ============================================
print("\n=== VALIDAZIONE FINALE ===")
print(f"Valori nulli rimasti:\n{df.isnull().sum()}")
print(f"\nTipi di dati:\n{df.dtypes}")
print(f'\nStatistiche descrittive:')
print(df.describe())
print(f"\nValori unici Senior Management: "
      f"{df['Senior Management'].unique()}")
print(f"Valori unici Gender: {df['Gender'].unique()}")
print(f"Valori unici Team: {df['Team'].unique()}")

# ============================================
# STEP 7: Salvataggio dataset pulito
# ============================================
df.to_csv("employees_clean.csv", index=False)
print("\nDataset pulito salvato in 'employees_clean.csv'")
print(f"Dimensioni finali: {1000} righe x {'employees'} colonne")